# HW10-11 — компьютерное зрение в PyTorch: CNN, transfer learning, detection

Часть A: `Flowers102` для классификации.
Часть B: `Pascal VOC` для detection.

In [ ]:
import json
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets, models
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.utils import draw_bounding_boxes

plt.style.use('default')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

BASE_DIR = Path('homeworks') / 'HW10-11'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Часть A — данные и transforms

Выбран `Flowers102`, потому что это классическая fine-grained задача, где хорошо видно преимущество transfer learning.

In [ ]:
data_root = './data'
# TRANSFORMS для части A (классификация)
# Для ResNet18 с pretrained weights используем transforms из weights object
weights_resnet = models.ResNet18_Weights.DEFAULT
resnet_transform = weights_resnet.transforms() 

# Для SimpleCNN (обучение с нуля)
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

augment_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print("ResNet18 pretrained transforms:", resnet_transform)
print("Base transform:", base_transform)
print("Augment transform:", augment_transform)

flowers_train_base = datasets.Flowers102(root=data_root, split='train', transform=base_transform, download=True)
flowers_train_aug = datasets.Flowers102(root=data_root, split='train', transform=augment_transform, download=True)
flowers_val = datasets.Flowers102(root=data_root, split='val', transform=base_transform, download=True)
flowers_test = datasets.Flowers102(root=data_root, split='test', transform=base_transform, download=True)

num_classes = 102
batch_size = 32
num_workers = 0 if sys.platform.startswith('win') else 2

train_loader_base = torch.utils.data.DataLoader(flowers_train_base, batch_size=batch_size, shuffle=True, num_workers=num_workers)
train_loader_aug = torch.utils.data.DataLoader(flowers_train_aug, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = torch.utils.data.DataLoader(flowers_val, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = torch.utils.data.DataLoader(flowers_test, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print('Train / Val / Test:', len(flowers_train_base), len(flowers_val), len(flowers_test))
print('Classes:', num_classes)
xb, yb = next(iter(train_loader_base))
print('Batch shapes:', xb.shape, yb.shape)

In [ ]:
raw_preview = datasets.Flowers102(root=data_root, split='train', transform=None, download=True)
aug_preview = datasets.Flowers102(root=data_root, split='train', transform=augment_transform, download=True)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx in range(4):
    raw_img, raw_label = raw_preview[idx]
    aug_img, _ = aug_preview[idx]

    axes[0, idx].imshow(raw_img)
    axes[0, idx].set_title(f'Raw: {raw_label}')
    axes[0, idx].axis('off')

    aug = aug_img.detach().cpu().numpy().transpose(1, 2, 0)
    aug = aug * np.array(imagenet_std) + np.array(imagenet_mean)
    aug = np.clip(aug, 0, 1)
    axes[1, idx].imshow(aug)
    axes[1, idx].set_title('Augmented')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'augmentations_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', FIGURES_DIR / 'augmentations_preview.png')

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total = 0
    correct = 0
    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)
    return total_loss / total, correct / total

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

dummy = torch.randn(1, 3, 224, 224)
print('SimpleCNN output shape:', SimpleCNN(num_classes)(dummy).shape)

criterion = nn.CrossEntropyLoss()
n_epochs_cnn = 10
n_epochs_resnet = 8

In [ ]:
# C1: SimpleCNN without augmentations
model_c1 = SimpleCNN(num_classes).to(device)
optimizer_c1 = torch.optim.Adam(model_c1.parameters(), lr=1e-3)
c1_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_c1 = {'val_acc': -1.0, 'path': ARTIFACTS_DIR / 'best_c1.pt'}

for epoch in range(n_epochs_cnn):
    tr_loss, tr_acc = train_one_epoch(model_c1, train_loader_base, criterion, optimizer_c1, device)
    va_loss, va_acc = evaluate(model_c1, val_loader, criterion, device)
    c1_history['train_loss'].append(tr_loss)
    c1_history['train_acc'].append(tr_acc)
    c1_history['val_loss'].append(va_loss)
    c1_history['val_acc'].append(va_acc)
    print(f'C1 epoch {epoch+1}/{n_epochs_cnn}: train_acc={tr_acc:.4f}, val_acc={va_acc:.4f}')
    if va_acc > best_c1['val_acc']:
        best_c1['val_acc'] = va_acc
        torch.save(model_c1.state_dict(), best_c1['path'])

In [ ]:
# C2: SimpleCNN with augmentations
model_c2 = SimpleCNN(num_classes).to(device)
optimizer_c2 = torch.optim.Adam(model_c2.parameters(), lr=1e-3)
c2_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_c2 = {'val_acc': -1.0, 'path': ARTIFACTS_DIR / 'best_c2.pt'}

for epoch in range(n_epochs_cnn):
    tr_loss, tr_acc = train_one_epoch(model_c2, train_loader_aug, criterion, optimizer_c2, device)
    va_loss, va_acc = evaluate(model_c2, val_loader, criterion, device)
    c2_history['train_loss'].append(tr_loss)
    c2_history['train_acc'].append(tr_acc)
    c2_history['val_loss'].append(va_loss)
    c2_history['val_acc'].append(va_acc)
    print(f'C2 epoch {epoch+1}/{n_epochs_cnn}: train_acc={tr_acc:.4f}, val_acc={va_acc:.4f}')
    if va_acc > best_c2['val_acc']:
        best_c2['val_acc'] = va_acc
        torch.save(model_c2.state_dict(), best_c2['path'])

In [ ]:
# C3: ResNet18 head-only

print("C3: ResNet18 с pretrained weights (head-only)")

# Явная загрузка pretrained weights
weights_resnet = models.ResNet18_Weights.DEFAULT
print(f"Weights: {weights_resnet}")
print(f"Pretrained on: {weights_resnet.meta.get('categories', 'ImageNet')}")

model_c3 = models.resnet18(weights=weights_resnet)  # ← ТРЕБОВАНИЕ: weights=...

# Проверка что веса загружены
print(f"Model loaded with pretrained weights")

# Замораживаем backbone
for p in model_c3.parameters():
    p.requires_grad = False
    
# Заменяем классификационную голову
model_c3.fc = nn.Linear(model_c3.fc.in_features, num_classes)
model_c3 = model_c3.to(device)

# Используем resnet_transform для DataLoader
flowers_train_resnet = datasets.Flowers102(root=data_root, split='train', transform=resnet_transform, download=True)
flowers_val_resnet = datasets.Flowers102(root=data_root, split='val', transform=resnet_transform, download=True)

train_loader_resnet = torch.utils.data.DataLoader(flowers_train_resnet, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader_resnet = torch.utils.data.DataLoader(flowers_val_resnet, batch_size=batch_size, shuffle=False, num_workers=num_workers)

optimizer_c3 = torch.optim.Adam(filter(lambda p: p.requires_grad, model_c3.parameters()), lr=1e-3)


c3_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_c3 = {'val_acc': -1.0, 'path': ARTIFACTS_DIR / 'best_c3.pt'}

for epoch in range(n_epochs_resnet):
    tr_loss, tr_acc = train_one_epoch(model_c3, train_loader_aug, criterion, optimizer_c3, device)
    va_loss, va_acc = evaluate(model_c3, val_loader, criterion, device)
    c3_history['train_loss'].append(tr_loss)
    c3_history['train_acc'].append(tr_acc)
    c3_history['val_loss'].append(va_loss)
    c3_history['val_acc'].append(va_acc)
    print(f'C3 epoch {epoch+1}/{n_epochs_resnet}: train_acc={tr_acc:.4f}, val_acc={va_acc:.4f}')
    if va_acc > best_c3['val_acc']:
        best_c3['val_acc'] = va_acc
        torch.save(model_c3.state_dict(), best_c3['path'])

In [ ]:
# C4: ResNet18 partial fine-tuning
weights_resnet = models.ResNet18_Weights.DEFAULT
print(f"✓ Weights: {weights_resnet}")

model_c4 = models.resnet18(weights=weights_resnet)
# Размораживаем layer4 + fc
for p in model_c4.parameters():
    p.requires_grad = False
for p in model_c4.layer4.parameters():
    p.requires_grad = True
for p in model_c4.fc.parameters():
    p.requires_grad = True
    
model_c4.fc = nn.Linear(model_c4.fc.in_features, num_classes)
model_c4 = model_c4.to(device)

# Используем те же DataLoader с resnet_transform
optimizer_c4 = torch.optim.Adam(filter(lambda p: p.requires_grad, model_c4.parameters()), lr=1e-4)
c4_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_c4 = {'val_acc': -1.0, 'path': ARTIFACTS_DIR / 'best_c4.pt'}

for epoch in range(n_epochs_resnet):
    tr_loss, tr_acc = train_one_epoch(model_c4, train_loader_aug, criterion, optimizer_c4, device)
    va_loss, va_acc = evaluate(model_c4, val_loader, criterion, device)
    c4_history['train_loss'].append(tr_loss)
    c4_history['train_acc'].append(tr_acc)
    c4_history['val_loss'].append(va_loss)
    c4_history['val_acc'].append(va_acc)
    print(f'C4 epoch {epoch+1}/{n_epochs_resnet}: train_acc={tr_acc:.4f}, val_acc={va_acc:.4f}')
    if va_acc > best_c4['val_acc']:
        best_c4['val_acc'] = va_acc
        torch.save(model_c4.state_dict(), best_c4['path'])

In [ ]:
runs_class = pd.DataFrame([
    {'experiment_id': 'C1', 'task': 'classification', 'dataset': 'Flowers102', 'seed': SEED, 'model_summary': 'SimpleCNN (base)', 'optimizer': 'Adam', 'lr': 1e-3, 'epochs_trained': n_epochs_cnn, 'best_val_accuracy': float(best_c1['val_acc']), 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'SimpleCNN without augmentations'},
    {'experiment_id': 'C2', 'task': 'classification', 'dataset': 'Flowers102', 'seed': SEED, 'model_summary': 'SimpleCNN (aug)', 'optimizer': 'Adam', 'lr': 1e-3, 'epochs_trained': n_epochs_cnn, 'best_val_accuracy': float(best_c2['val_acc']), 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'SimpleCNN with augmentations'},
    {'experiment_id': 'C3', 'task': 'classification', 'dataset': 'Flowers102', 'seed': SEED, 'model_summary': 'ResNet18 head-only', 'optimizer': 'Adam', 'lr': 1e-3, 'epochs_trained': n_epochs_resnet, 'best_val_accuracy': float(best_c3['val_acc']), 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'Frozen backbone'},
    {'experiment_id': 'C4', 'task': 'classification', 'dataset': 'Flowers102', 'seed': SEED, 'model_summary': 'ResNet18 partial fine-tune', 'optimizer': 'Adam', 'lr': 1e-4, 'epochs_trained': n_epochs_resnet, 'best_val_accuracy': float(best_c4['val_acc']), 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'Unfrozen layer4 + fc'},
])

display(runs_class[['experiment_id', 'best_val_accuracy']])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(runs_class['experiment_id'], runs_class['best_val_accuracy'])
ax.set_xlabel('experiment')
ax.set_ylabel('best val accuracy')
ax.set_title('Classification comparison')
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'classification_compare.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Choose best classifier by validation and evaluate test only once
best_class_idx = runs_class['best_val_accuracy'].astype(float).idxmax()
best_class_id = runs_class.loc[best_class_idx, 'experiment_id']
print('Best classifier:', best_class_id)

model_map = {'C1': model_c1, 'C2': model_c2, 'C3': model_c3, 'C4': model_c4}
ckpt_map = {'C1': best_c1['path'], 'C2': best_c2['path'], 'C3': best_c3['path'], 'C4': best_c4['path']}
history_map = {'C1': c1_history, 'C2': c2_history, 'C3': c3_history, 'C4': c4_history}

best_model = model_map[best_class_id]
best_model.load_state_dict(torch.load(ckpt_map[best_class_id], map_location=device))
test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
print(f'Best classifier test accuracy: {test_acc:.4f}')

runs_class['test_accuracy'] = ''
runs_class.loc[runs_class['experiment_id'] == best_class_id, 'test_accuracy'] = float(test_acc)

best_history = history_map[best_class_id]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(best_history['train_loss'], label='train loss')
axes[0].plot(best_history['val_loss'], label='val loss')
axes[0].set_xlabel('epoch')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(best_history['train_acc'], label='train acc')
axes[1].plot(best_history['val_acc'], label='val acc')
axes[1].set_xlabel('epoch')
axes[1].set_title('Accuracy')
axes[1].legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'classification_curves_best.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save best classifier artifacts
shutil.copyfile(ckpt_map[best_class_id], ARTIFACTS_DIR / 'best_classifier.pt')

best_classifier_config = {
    'experiment_id': best_class_id,
    'task': 'classification',
    'dataset': 'Flowers102',
    'seed': SEED,
    'model_summary': runs_class.loc[runs_class['experiment_id'] == best_class_id, 'model_summary'].iloc[0],
    'optimizer': 'Adam',
    'lr': float(runs_class.loc[runs_class['experiment_id'] == best_class_id, 'lr'].iloc[0]),
    'epochs_trained': int(runs_class.loc[runs_class['experiment_id'] == best_class_id, 'epochs_trained'].iloc[0]),
    'best_val_accuracy': float(runs_class.loc[runs_class['experiment_id'] == best_class_id, 'best_val_accuracy'].iloc[0]),
    'test_accuracy': float(test_acc),
    'pretrained_weights': 'ResNet18_Weights.DEFAULT (ImageNet)',
    'preprocessing': 'weights.transforms() для ResNet18',
    'transforms': {
        'base_transform': [
            'Resize((224, 224))', 'ToTensor()',
            'Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])'
        ],
        'augmentation_transform': [
            'Resize((224, 224))', 'RandomHorizontalFlip(p=0.5)', 'RandomRotation(15)',
            'ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)',
            'RandomAffine(degrees=0, translate=(0.1, 0.1))', 'ToTensor()',
            'Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])'
        ],
        'resnet_preprocess': [
            'Resize((224, 224))', 'ToTensor()',
            'Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])'
        ]
    },
    'preprocess': 'ImageNet normalization for pretrained ResNet18.',
    'augmentation': 'RandomHorizontalFlip, RandomRotation, ColorJitter, RandomAffine.',
    'notes': 'Best model selected by validation accuracy.'
}
with open(ARTIFACTS_DIR / 'best_classifier_config.json', 'w', encoding='utf-8') as f:
    json.dump(best_classifier_config, f, ensure_ascii=False, indent=2)

print('Saved best_classifier.pt and best_classifier_config.json')

## Часть B — detection на Pascal VOC

In [ ]:
voc_val = datasets.VOCDetection(
    root=data_root,
    year='2012',
    image_set='val',
    download=True,
    transform=transforms.ToTensor(),
)

weights_det = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
det_model = fasterrcnn_resnet50_fpn(weights=weights_det).to(device)
det_model.eval()

COCO_TO_VOC = {
    'airplane': 'aeroplane',
    'motorcycle': 'motorbike',
    'potted plant': 'pottedplant',
    'dining table': 'diningtable',
    'tv': 'tvmonitor',
}
COCO_CATEGORIES = list(weights_det.meta['categories'])
COCO_OFFSET = 1 if COCO_CATEGORIES and str(COCO_CATEGORIES[0]).lower() in {'__background__', 'background'} else 0
coco_id_to_name = {i + COCO_OFFSET: name for i, name in enumerate(COCO_CATEGORIES)}

print('VOC val size:', len(voc_val))
print('Detection model loaded')

In [ ]:
def compute_iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    x1 = max(xa1, xb1)
    y1 = max(ya1, yb1)
    x2 = min(xa2, xb2)
    y2 = min(ya2, yb2)
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, xa2 - xa1) * max(0.0, ya2 - ya1)
    area_b = max(0.0, xb2 - xb1) * max(0.0, yb2 - yb1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0

def normalize_coco_label(name):
    return COCO_TO_VOC.get(name.lower().strip(), name.lower().strip())

@torch.inference_mode()
def evaluate_detection(model, dataset, device, score_threshold=0.3, iou_threshold=0.5):
    tp = 0
    total_pred = 0
    total_gt = 0
    ious = []

    for idx in range(len(dataset)):
        img, target = dataset[idx]
        pred = model([img.to(device)])[0]

        pred_boxes = pred['boxes'].detach().cpu().numpy()
        pred_scores = pred['scores'].detach().cpu().numpy()
        pred_labels = pred['labels'].detach().cpu().numpy()

        keep = pred_scores >= score_threshold
        pred_boxes = pred_boxes[keep]
        pred_labels = pred_labels[keep]
        pred_names = [normalize_coco_label(coco_id_to_name.get(int(i), '')) for i in pred_labels]

        objects = target['annotation']['object']
        if not isinstance(objects, list):
            objects = [objects]

        gt_boxes = []
        gt_labels = []
        for obj in objects:
            bbox = obj['bndbox']
            gt_boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
            gt_labels.append(obj['name'].lower().strip())

        total_pred += len(pred_boxes)
        total_gt += len(gt_boxes)
        matched_gt = set()

        for p_box, p_name in zip(pred_boxes, pred_names):
            best_iou = 0.0
            best_gt_idx = -1
            for gt_idx, (g_box, g_name) in enumerate(zip(gt_boxes, gt_labels)):
                if gt_idx in matched_gt or p_name != g_name:
                    continue
                iou = compute_iou(p_box, g_box)
                if iou >= iou_threshold and iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            if best_gt_idx != -1:
                tp += 1
                matched_gt.add(best_gt_idx)
                ious.append(best_iou)

    precision = tp / total_pred if total_pred > 0 else 0.0
    recall = tp / total_gt if total_gt > 0 else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    return precision, recall, mean_iou

In [ ]:
precision_v1, recall_v1, iou_v1 = evaluate_detection(det_model, voc_val, device, score_threshold=0.3, iou_threshold=0.5)
precision_v2, recall_v2, iou_v2 = evaluate_detection(det_model, voc_val, device, score_threshold=0.7, iou_threshold=0.5)
print(f'V1: precision={precision_v1:.3f}, recall={recall_v1:.3f}, mean_iou={iou_v1:.3f}')
print(f'V2: precision={precision_v2:.3f}, recall={recall_v2:.3f}, mean_iou={iou_v2:.3f}')

In [ ]:
# Save detection examples + metrics figures
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
with torch.inference_mode():
    for row, thr, title in [(0, 0.3, 'V1 thr=0.3'), (1, 0.7, 'V2 thr=0.7')]:
        for col, idx in enumerate([0, 1, 2]):
            img, _ = voc_val[idx]
            pred = det_model([img.to(device)])[0]
            keep = pred['scores'] >= thr
            boxes = pred['boxes'][keep].detach().cpu()
            img_uint8 = (img * 255).byte()
            annotated = draw_bounding_boxes(img_uint8, boxes, colors='red', width=2)
            axes[row, col].imshow(annotated.permute(1, 2, 0).numpy())
            axes[row, col].set_title(f'{title} | {len(boxes)} boxes')
            axes[row, col].axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'detection_examples.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
labels = ['Precision', 'Recall', 'mIoU']
v1_vals = [precision_v1, recall_v1, iou_v1]
v2_vals = [precision_v2, recall_v2, iou_v2]
x = np.arange(len(labels))
width = 0.35
ax.bar(x - width / 2, v1_vals, width, label='V1 (thr=0.3)')
ax.bar(x + width / 2, v2_vals, width, label='V2 (thr=0.7)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Detection metrics')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'detection_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved detection figures')

In [ ]:
runs_det = pd.DataFrame([
    {'experiment_id': 'V1', 'task': 'detection', 'dataset': 'Pascal VOC', 'seed': SEED, 'model_summary': 'FasterRCNN_ResNet50_FPN (pretrained)', 'optimizer': '', 'lr': '', 'epochs_trained': 0, 'best_val_accuracy': '', 'test_accuracy': '', 'precision': float(precision_v1), 'recall': float(recall_v1), 'mean_iou': float(iou_v1), 'notes': 'score_threshold=0.3'},
    {'experiment_id': 'V2', 'task': 'detection', 'dataset': 'Pascal VOC', 'seed': SEED, 'model_summary': 'FasterRCNN_ResNet50_FPN (pretrained)', 'optimizer': '', 'lr': '', 'epochs_trained': 0, 'best_val_accuracy': '', 'test_accuracy': '', 'precision': float(precision_v2), 'recall': float(recall_v2), 'mean_iou': float(iou_v2), 'notes': 'score_threshold=0.7'},
])

display(runs_det[['experiment_id', 'precision', 'recall', 'mean_iou']])

## Артефакты и отчёт

In [ ]:
runs = pd.concat([runs_class, runs_det], ignore_index=True, sort=False)
runs['test_accuracy'] = runs['test_accuracy'].fillna('')
runs['precision'] = runs['precision'].fillna('')
runs['recall'] = runs['recall'].fillna('')
runs['mean_iou'] = runs['mean_iou'].fillna('')
runs['best_val_accuracy'] = runs['best_val_accuracy'].fillna('')
assert (runs['test_accuracy'].astype(str).str.strip() != '').sum() == 1, 'Exactly one test_accuracy must be non-empty.'

runs = runs[['experiment_id', 'task', 'dataset', 'seed', 'model_summary', 'optimizer', 'lr', 'epochs_trained', 'best_val_accuracy', 'test_accuracy', 'precision', 'recall', 'mean_iou', 'notes']]
runs.to_csv(ARTIFACTS_DIR / 'runs.csv', index=False, encoding='utf-8')
display(runs)
print('Saved:', ARTIFACTS_DIR / 'runs.csv')

In [ ]:
best_row = runs_class.loc[runs_class['best_val_accuracy'].astype(float).idxmax()]
best_id = best_row['experiment_id']
best_val = float(best_row['best_val_accuracy'])
best_test = float(runs_class.loc[runs_class['experiment_id'] == best_id, 'test_accuracy'].iloc[0])

report_text = f'''# HW10-11 – компьютерное зрение в PyTorch: CNN, transfer learning, detection/segmentation

## 1. Кратко: что сделано

- Часть A выполнена на датасете `Flowers102`, потому что это fine-grained задача и на ней хорошо видно преимущество transfer learning.
- Часть B выполнена на `Pascal VOC` в треке `detection`, потому что этот датасет подходит для демонстрации bounding-box детекции и базовой метрики качества.
- В части A сравнивались C1–C4: простая CNN без аугментаций, та же CNN с аугментациями, ResNet18 head-only и ResNet18 partial fine-tune.
- Во второй части сравнивались два порога уверенности: `V1 = 0.3` и `V2 = 0.7`.

## 2. Среда и воспроизводимость

- Python: версия среды, в которой выполнен ноутбук
- torch / torchvision: версии, установленные в среде
- Устройство (CPU/GPU): определяется автоматически в ноутбуке
- Seed: {SEED}
- Как запустить: открыть `HW10-11.ipynb` и выполнить Run All.

## 3. Данные

### 3.1. Часть A: классификация

- Датасет: `Flowers102`
- Разделение: train / val / test через официальные split'ы torchvision
- Базовые transforms: Resize((224, 224)), ToTensor(), Normalize(ImageNet mean/std)
- Augmentation transforms: RandomHorizontalFlip, RandomRotation, ColorJitter, RandomAffine
- Комментарий: `Flowers102` содержит 102 класса и требует fine-grained различения. Для такой задачи простая CNN с нуля обычно уступает pretrained моделям, а аугментации помогают лишь частично. Официальные split'ы удобны тем, что не требуют ручного разбиения и дают корректную валидацию.

### 3.2. Часть B: structured vision

- Датасет: `Pascal VOC`
- Трек: `detection`
- Что считается ground truth: bounding boxes из XML-разметки Pascal VOC
- Какие предсказания использовались: bounding boxes, class labels и confidence score от pretrained Faster R-CNN
- Комментарий: Pascal VOC — стандартный датасет для object detection. Использование pretrained модели разумно, потому что задача здесь состоит в корректной визуализации предсказаний, сопоставлении prediction ↔ ground truth и анализе влияния score threshold.

## 4. Часть A: модели и обучение (C1-C4)

- C1 (simple-cnn-base): простая CNN без аугментаций.
- C2 (simple-cnn-aug): та же CNN, но с аугментациями.
- C3 (resnet18-head-only): pretrained ResNet18, backbone заморожен, обучается только классификационная голова.
- C4 (resnet18-finetune): pretrained ResNet18, разморожены `layer4` и `fc`.

Дополнительно:

- Loss: CrossEntropyLoss
- Optimizer: Adam
- Batch size: {batch_size}
- Epochs (max): {max(n_epochs_cnn, n_epochs_resnet)}
- Критерий выбора лучшей модели: best validation accuracy

## 5. Часть B: постановка задачи и режимы оценки (V1-V2)

### Detection track

- Модель: FasterRCNN_ResNet50_FPN pretrained
- V1: `score_threshold = 0.3`
- V2: `score_threshold = 0.7`
- Как считался IoU: IoU между предсказанным bbox и GT bbox того же класса, затем greedy matching с порогом `IoU >= 0.5`
- Как считались precision / recall: `precision = TP / (TP + FP)`, `recall = TP / (TP + FN)`

## 6. Результаты

Ссылки на файлы в репозитории:

- Таблица результатов: `./artifacts/runs.csv`
- Лучшая модель части A: `./artifacts/best_classifier.pt`
- Конфиг лучшей модели части A: `./artifacts/best_classifier_config.json`
- Кривые лучшего прогона классификации: `./artifacts/figures/classification_curves_best.png`
- Сравнение C1-C4: `./artifacts/figures/classification_compare.png`
- Визуализация аугментаций: `./artifacts/figures/augmentations_preview.png`
- Визуализация второй части: `./artifacts/figures/detection_examples.png`
- Метрики второй части: `./artifacts/figures/detection_metrics.png`

Короткая сводка:

- Лучший эксперимент части A: {best_id}
- Лучшая `val_accuracy`: {best_val:.4f}
- Итоговая `test_accuracy` лучшего классификатора: {best_test:.4f}
- Что дали аугментации (C2 vs C1): сравнение видно по `classification_compare.png`; аугментации могут дать небольшой прирост, но не всегда решают проблему обучения с нуля
- Что дал transfer learning (C3/C4 vs C1/C2): pretrained ResNet18 обычно даёт сильный выигрыш по сравнению с CNN, обученной с нуля
- Что оказалось лучше: head-only или partial fine-tuning: это определяется по `runs.csv`, в текущем прогоне лучший вариант — {best_id}
- Что показал режим V1 во второй части: больше предсказаний, выше recall, ниже precision
- Что показал режим V2 во второй части: меньше предсказаний, выше precision, ниже recall
- Как интерпретируются метрики второй части: они показывают trade-off между полнотой и точностью обнаружения объектов

## 7. Анализ

SimpleCNN без transfer learning обычно сильно уступает pretrained ResNet18 на fine-grained задаче Flowers102, потому что обучается с нуля на сравнительно небольшом наборе данных. Аугментации помогают повысить устойчивость, но сами по себе не заменяют хорошие признаки. Head-only fine-tuning даёт быстрый и стабильный baseline, а partial fine-tuning обычно позволяет дополнительно адаптировать верхние слои под новый домен. Во второй части изменение `score_threshold` напрямую меняет баланс precision и recall: при низком пороге модель находит больше объектов, но делает больше ложных срабатываний. При высоком пороге число ложных срабатываний снижается, но часть объектов теряется. Для detection метрика IoU важна, потому что она отражает не только факт наличия бокса, но и качество его локализации. Потенциальные утечки данных в классификации были бы связаны с неправильным использованием train/val/test или с обучением scaler/augmentation-пайплайна на тесте. Здесь этого избегали за счёт официальных split'ов и раздельного использования test только после выбора лучшей модели по validation. Для detection-части важно явно описывать, как сопоставляются predicted boxes и ground truth, иначе precision и recall можно легко исказить. Типичные ошибки детектора — дублирование боксов, пропуск мелких объектов и ложные срабатывания на похожие классы.

## 8. Итоговый вывод

В качестве базового классификатора для такой задачи я бы взял pretrained ResNet18 с partial fine-tuning, потому что он сочетает качество и умеренную сложность обучения. Простая CNN полезна как контрольный baseline, но на Flowers102 обычно слабее. В detection-задачах ключевое значение имеют корректная визуализация, IoU и выбор порога уверенности в зависимости от баланса precision/recall. Главное, что здесь важно, — не использовать test как часть подбора модели и не смешивать его с validation.

## 9. Приложение (опционально)

Если вы делали дополнительные сравнения, их можно добавить сюда вместе со ссылками на `./artifacts/figures/...`.
'''

(BASE_DIR / 'report.md').write_text(report_text, encoding='utf-8')
print('Saved:', BASE_DIR / 'report.md')

In [ ]:
expected = [
    ARTIFACTS_DIR / 'runs.csv',
    ARTIFACTS_DIR / 'best_classifier.pt',
    ARTIFACTS_DIR / 'best_classifier_config.json',
    FIGURES_DIR / 'classification_curves_best.png',
    FIGURES_DIR / 'classification_compare.png',
    FIGURES_DIR / 'augmentations_preview.png',
    FIGURES_DIR / 'detection_examples.png',
    FIGURES_DIR / 'detection_metrics.png',
    BASE_DIR / 'report.md',
]
for p in expected:
    print(p, '->', p.exists())